# Bug-Algorithmus

Jonas Frei, 15.10.2026

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.path import Path

## Hilfsfunktionen

Funktion zur Berechnung des nächstgelegenen Punktes und dessen Abstand zwischen einem Punkt und einem Segment einer Linie mit Endpunkten P1 und P2

In [ ]:
def point_to_segment_distance(point, P1, P2):
    point = np.asarray(point, dtype=float)
    P1 = np.asarray(P1, dtype=float)
    P2 = np.asarray(P2, dtype=float)
    l2 = np.linalg.norm(P1 - P2)**2
    if l2 == 0:
        return P1, np.linalg.norm(point - P1)
    t = max(0, min(1, np.dot(point - P1, P2 - P1)/l2))
    projection = P1 + t*(P2 - P1)
    return projection, np.linalg.norm(point - projection)

Funktion um zu Prüfen, ob ein Punkt innerhalb eines Hindernis liegt

In [ ]:
def is_inside_obstacle(point, obstacles):
    for obstacle in obstacles:
        if Path(obstacle).contains_point(point):
            return True
    return False

Funktion zum Berechnen des nächstgelegenen Punktes und dessen Abstand zwischen der aktuellen Position und eines Hindernis

In [ ]:
def find_closest_point_on_boundary(current_pos, obstacles):
    min_dist = np.inf
    closest_point = current_pos
    for obstacle in obstacles:
        n = obstacle.shape[0]
        for k in range(n):
            next_idx = (k + 1) % n
            cp, dist = point_to_segment_distance(current_pos, obstacle[k], obstacle[next_idx])
            if dist < min_dist:
                min_dist = dist
                closest_point = cp
    return closest_point, min_dist

## Initialisierung

**Start- und Zielposition**

In [ ]:
start = np.array([1.0, 1.0])
target = np.array([10.0, 10.0])

**Hindernisse (Liste von Polygonen)**

In [ ]:
obstacles = [
    np.array([[3, 2], [5, 2], [5, 4], [3, 4]], dtype=float),                    # Hindernis 1
    np.array([[7, 6], [8, 6], [8, 9], [6, 9]], dtype=float),                    # Hindernis 2
    np.array([[-0.5, 3.5], [3.5, 3.5], [3.5, 4], [-0.5, 4]], dtype=float),      # Hindernis 3
    np.array([[-0.5, -0.5], [0, -0.5], [0, 4], [-0.5, 4]], dtype=float),        # Hindernis 4
    np.array([[-0.5, -0.5], [2.5, -0.5], [2.5, 0], [-0.5, 0]], dtype=float),    # Hindernis 5
    np.array([[2, -0.5], [2.5, -0.5], [2.5, 2.5], [2, 2.5]], dtype=float),      # Hindernis 6
]

## Plotten der Umgebung

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.grid(True)
ax.set_aspect('equal')

ax.plot(start[0], start[1], 'go', markersize=10, markerfacecolor='g', label="Startposition")
ax.plot(target[0], target[1], 'ro', markersize=10, markerfacecolor='r', label="Zielposition")

for i, obstacle in enumerate(obstacles, start=1):
    ax.fill(obstacle[:, 0], obstacle[:, 1], 'k', label=f"Hindernis #{i}")

ax.plot([start[0], target[0]], [start[1], target[1]], 'c--', linewidth=2, label="Verbindungsvektor")

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("Bug-Algorithmus")
ax.legend(loc="best")

## Bug-Algorithmus

In [ ]:
current_pos = start.copy()
lineStartTarget = target - start
path = [current_pos.copy()]
state = 'move_to_target'
iter_ = 0

while np.linalg.norm(current_pos - target) > 0.1:
    if state == 'move_to_target':
        # Berechne die Bewegung entlang der Linie in Richtung des Ziels
        direction = lineStartTarget/np.linalg.norm(lineStartTarget)
        next_pos = current_pos + 0.1*direction
        # Führe die Bewegung aus, falls kein Hindernis im Weg
        if not is_inside_obstacle(next_pos, obstacles):
            current_pos = next_pos
            path.append(current_pos.copy())
        else:
            # Wechsel auf "Wall-Follow" Modus und speichere aktuelle Position,
            # falls ein Hindernis im Weg
            hit_point = current_pos.copy()
            state = 'follow_wall'
            iter_ = 0

    elif state == 'follow_wall':
        iter_ += 1
        if iter_ > 500:
            print('Error')
            break
        # Folge dem Hindernisses im Gegenuhrzeigersinn
        cp, dist = find_closest_point_on_boundary(current_pos, obstacles)
        direction = (cp - current_pos)/np.linalg.norm(cp - current_pos)
        next_pos = current_pos + 0.1*np.array([-direction[1], direction[0]]) + (dist - 0.1)*direction
        current_pos = next_pos
        path.append(current_pos.copy())

        # Wechsel auf normalen Modus, falls wieder zurück auf der Linie
        # und näher am Ziel
        _, dist = point_to_segment_distance(next_pos, start, target)
        if dist < 0.1 and np.linalg.norm(next_pos - target) < np.linalg.norm(hit_point - target):
            state = 'move_to_target'
else:
    print('Zielposition erreicht!')

path = np.array(path)

## Plotten des gesamten Pfades

In [ ]:
ax.plot(path[:, 0], path[:, 1], 'b-', linewidth=2, label="Pfad")
ax.legend(loc="best")
fig

## Animation des Pfades

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter

fig_anim, ax_anim = plt.subplots(figsize=(8, 8))
ax_anim.grid(True)
ax_anim.set_aspect('equal')
ax_anim.set_xlim(-1, 11)
ax_anim.set_ylim(-1, 11)

ax_anim.plot(start[0], start[1], 'go', markersize=10, label="Startposition")
ax_anim.plot(target[0], target[1], 'ro', markersize=10, label="Zielposition")
for i, obstacle in enumerate(obstacles, start=1):
    ax_anim.fill(obstacle[:, 0], obstacle[:, 1], 'k', label=f"Hindernis #{i}" if i == 1 else None)
ax_anim.plot([start[0], target[0]], [start[1], target[1]], 'c--', linewidth=2, label="Verbindungsvektor")

ax_anim.set_xlabel("X")
ax_anim.set_ylabel("Y")
ax_anim.set_title("Bug-Algorithmus (Animation)")
ax_anim.legend(loc="upper left")

path_line, = ax_anim.plot([], [], 'b-', linewidth=2)
robot_dot, = ax_anim.plot([], [], 'bo', markersize=8)


def update(frame):
    path_line.set_data(path[:frame + 1, 0], path[:frame + 1, 1])
    robot_dot.set_data([path[frame, 0]], [path[frame, 1]])
    return path_line, robot_dot


anim = FuncAnimation(fig_anim, update, frames=len(path), interval=20, blit=True)
anim.save('bug_algorithmus_animation.gif', writer=PillowWriter(fps=30))
plt.close(fig_anim)

In [ ]:
from IPython.display import Image

Image(filename='bug_algorithmus_animation.gif')